# Dump Hamiltonian to .npz

In [17]:
import numpy as np
import jax
import jax.numpy as jnp
from jax import jit
from functools import lru_cache

In [18]:
jax.config.update("jax_enable_x64", True)

In [3]:
pauli_mat = jnp.eye(4)
class Pauli:
        r"""Configuration manager for Pauli string observables.
            
        Examples
        --------
        >>> config = PauliStringConfig(n_q=4, n_p=2)
        >>> config.add(0, "ZZII", 0.5)  # Add 0.5 * Z⊗Z⊗I⊗I
        >>> config.add(1, "XIXY", -0.4) # Add -0.4 * X⊗I⊗X⊗Y
        """
        
        def __init__(self, n_q: int, n_p: int):
            self.n_q = n_q
            self.n_p = n_p
            self.mat = jnp.zeros((self.n_p, self.n_q, 4))
            self.weights = jnp.ones(self.n_p)
            self.sync()

        def sync(self):
            self.tn =  self.mat @ pauli_mat

        @property
        def w(self):
            return self.weights
        
        @w.setter
        def w(self, value):
            self.weights = value

        @property
        def data(self):
            r"""Return (tn, w) tuple."""
            return (self.tn, self.w)
        
        def add(self, index: int, pauli_string: str, weight: float):
                p_str = pauli_string.strip().upper()

                if len(p_str) != self.n_q:
                    raise ValueError(
                        f"Pauli string length ({len(p_str)}) does not match "
                        f"num_qubits ({self.n_q})."
                    )
                
                mapping = {'I': 0, 'X': 1, 'Y': 2, 'Z': 3}
                
                indices = []
                for char in p_str:
                    if char not in mapping:
                        raise ValueError(
                            f"Invalid character '{char}' in Pauli string. "
                            f"Only I, X, Y, Z are supported."
                        )
                    indices.append(mapping[char])
                
                row_values = jax.nn.one_hot(jnp.array(indices), 4)
            
                self.mat = self.mat.at[index].set(row_values)
                self.weights = self.weights.at[index].set(weight)
                self.sync()

        def save(self, path: str):
                """Save Pauli data to a .npz file."""
                np.savez(path, mat=np.asarray(self.mat), w=np.asarray(self.w))
        
        @classmethod
        def load(cls, path: str) -> 'Pauli':
                """Load Pauli data from a .npz file."""
                data = np.load(path)
                mat, w = data['mat'], data['w']
                n_p, n_q = mat.shape[0], mat.shape[1]
                obj = cls(n_q=n_q, n_p=n_p)
                obj.mat = jnp.array(mat)
                obj.w = jnp.array(w)
                obj.sync()
                return obj

E0325 10:42:08.467668  449764 cuda_executor.cc:1743] Could not get kernel mode driver version: [INVALID_ARGUMENT: Version does not match the format X.Y.Z]
E0325 10:42:08.479111  449670 cuda_executor.cc:1743] Could not get kernel mode driver version: [INVALID_ARGUMENT: Version does not match the format X.Y.Z]


In [19]:
import h5py

# 你想查看的数据集名称，你可以随时改成 'ham_BK-8', 'ham_molec-4' 等等
dataset_name = 'ham_JW-8' 

with h5py.File('./ham_hdf5/H2.hdf5', 'r') as f:
    dataset = f[dataset_name]
    data = dataset[()] # type: ignore
    
    print(f"=== {dataset_name} 的数据类型: {type(data)} ===")
    if isinstance(data, bytes):
        try:
            print(data.decode('utf-8'))
        except UnicodeDecodeError:
            print("数据是二进制字节，可能使用了 pickle 等方式序列化。")
            print(data)
    else:
        print(data)

=== ham_JW-8 的数据类型: <class 'bytes'> ===
(1.306270932073354+0j) [] +
(-0.014034099995004047+0j) [X0 X1 Y2 Y3] +
(-0.017163359706996492+0j) [X0 X1 Y2 Z3 Z4 Z5 Z6 Y7] +
(-0.017163359706996492+0j) [X0 X1 X3 Z4 Z5 X6] +
(-0.021241221323288813+0j) [X0 X1 Y4 Y5] +
(-0.029791404539838855+0j) [X0 X1 Y6 Y7] +
(0.014034099995004047+0j) [X0 Y1 Y2 X3] +
(0.017163359706996492+0j) [X0 Y1 Y2 Z3 Z4 Z5 Z6 X7] +
(-0.017163359706996492+0j) [X0 Y1 Y3 Z4 Z5 X6] +
(0.021241221323288813+0j) [X0 Y1 Y4 X5] +
(0.029791404539838855+0j) [X0 Y1 Y6 X7] +
(0.00602624511952636+0j) [X0 Z1 X2 X3 Z4 X5] +
(0.00602624511952636+0j) [X0 Z1 X2 Y3 Z4 Y5] +
(-0.011825396144328048+0j) [X0 Z1 X2 X4 Z5 X6] +
(0.001277911257992757+0j) [X0 Z1 X2 Y4 Z5 Y6] +
(-0.012080304750921995+0j) [X0 Z1 X2 X5 Z6 X7] +
(-0.012080304750921995+0j) [X0 Z1 X2 Y5 Z6 Y7] +
(-0.013103307402320803+0j) [X0 Z1 Y2 Y4 Z5 X6] +
(0.01335821600891475+0j) [X0 Z1 Z2 X3 Y4 Z5 Z6 Y7] +
(0.00025490860659394725+0j) [X0 Z1 Z2 X3 X5 X6] +
(-0.01335821600891475+0j) [X0

In [20]:

import re

raw = data.decode('utf-8') if isinstance(data, bytes) else data

pattern = r'\(([^)]+)\)\s*\[([^\]]*)\]'
terms = []
for match in re.finditer(pattern, raw):
    coeff = float(match.group(1).replace('+0j', '').replace('+0', ''))
    ops_str = match.group(2).strip()
    
    if not ops_str:
        continue
    
    pauli_map = {}
    for op in ops_str.split():
        gate = op[0]
        qubit = int(op[1:])
        pauli_map[qubit] = gate
    
    pauli_str = ''
    for q in range(4):
        pauli_str += pauli_map.get(q, 'I')
    
    terms.append((pauli_str, coeff))
    
n_q = 4
n_p = len(terms)
ham = Pauli(n_q=n_q, n_p=n_p)

for i, (pstr, w) in enumerate(terms):
    ham.add(i, pstr, w)
    print(f"  [{i}] {pstr}  weight={w}")

print(f"\n共 {n_p} 个 Pauli 项 (已省略单位算符)")

  [0] XXYY  weight=-0.014034099995004047
  [1] XXYZ  weight=-0.017163359706996492
  [2] XXIX  weight=-0.017163359706996492
  [3] XXII  weight=-0.021241221323288813
  [4] XXII  weight=-0.029791404539838855
  [5] XYYX  weight=0.014034099995004047
  [6] XYYZ  weight=0.017163359706996492
  [7] XYIY  weight=-0.017163359706996492
  [8] XYII  weight=0.021241221323288813
  [9] XYII  weight=0.029791404539838855
  [10] XZXX  weight=0.00602624511952636
  [11] XZXY  weight=0.00602624511952636
  [12] XZXI  weight=-0.011825396144328048
  [13] XZXI  weight=0.001277911257992757
  [14] XZXI  weight=-0.012080304750921995
  [15] XZXI  weight=-0.012080304750921995
  [16] XZYI  weight=-0.013103307402320803
  [17] XZZX  weight=0.01335821600891475
  [18] XZZX  weight=0.00025490860659394725
  [19] XZZY  weight=-0.01335821600891475
  [20] XZZY  weight=0.00025490860659394725
  [21] XZZZ  weight=-0.049949991390849664
  [22] XZZZ  weight=0.01883837207821714
  [23] XZZZ  weight=0.014651488437623306
  [24] XZZZ  we

In [21]:
import re
import h5py

def load_pauli_from_hdf5(filepath: str, dataset_name: str) -> Pauli:
    """Load a Pauli Hamiltonian from an HDF5 file. Identity-only terms are skipped."""
    with h5py.File(filepath, 'r') as f:
        raw = f[dataset_name][()]
    if isinstance(raw, bytes):
        raw = raw.decode('utf-8')

    pattern = r'\(([^)]+)\)\s*\[([^\]]*)\]'
    terms = []
    for match in re.finditer(pattern, raw):
        coeff = float(match.group(1).replace('+0j', '').replace('-0j', ''))
        ops_str = match.group(2).strip()
        if not ops_str:
            continue

        pauli_map = {}
        n_q = 0
        for op in ops_str.split():
            qubit = int(op[1:])
            pauli_map[qubit] = op[0]
            n_q = max(n_q, qubit + 1)
        terms.append((pauli_map, coeff, n_q))

    n_q = max(t[2] for t in terms)
    ham = Pauli(n_q=n_q, n_p=len(terms))
    for i, (pauli_map, w, _) in enumerate(terms):
        pauli_str = ''.join(pauli_map.get(q, 'I') for q in range(n_q))
        ham.add(i, pauli_str, w)
    return ham

In [25]:
ham = load_pauli_from_hdf5(filepath='./ham_hdf5/H2.hdf5', dataset_name='ham_JW-4')
ham.save('./ham/H2_JW4.npz')

In [ ]:
ham = load_pauli_from_hdf5(filepath='./ham_hdf5/H2.hdf5', dataset_name='ham_JW-8')
ham.save('./ham/H2_JW8.npz')

In [31]:
ham = load_pauli_from_hdf5(filepath='./ham_hdf5/LiH.hdf5', dataset_name='ham_JW-12')
ham.save('./ham/LiH_JW12.npz')

In [39]:
ham = load_pauli_from_hdf5(filepath='./ham_hdf5/BeH.hdf5', dataset_name='ham_JW12')
ham.save('./ham/BeH_JW12.npz')

In [42]:
ham = load_pauli_from_hdf5(filepath='./ham_hdf5/NH.hdf5', dataset_name='ham_JW-14')
ham.save('./ham/NH_JW14.npz')

In [ ]:
ham = load_pauli_from_hdf5(filepath='./ham_hdf5/NH.hdf5', dataset_name='ham_JW-14')
ham.save('./ham/NH_JW14.npz')

# 检查 HDF5 文件中的数据类型

In [29]:
import h5py
from pathlib import Path

def show_types(filename: str):
    """给出文件名，输出该 HDF5 文件中包含的 dataset 原始 key 名。"""
    if not filename.endswith('.hdf5'):
        filename += '.hdf5'
    filepath = Path('./ham_hdf5') / filename
    
    if not filepath.exists():
        print(f'文件不存在: {filepath}')
        print(f'可用文件: {[p.stem for p in sorted(Path("./ham_hdf5").glob("*.hdf5"))]}')
        return
    
    with h5py.File(filepath, 'r') as f:
        keys = sorted(f.keys())
    
    # 按类别分组，保留原始 key 名
    categories = {}
    for key in keys:
        for prefix in ['ham_JW', 'ham_BK', 'ham_parity', 'ham_molec']:
            if key.startswith(prefix):
                categories.setdefault(prefix, []).append(key)
                break
    
    print(f'📄 {filepath.stem}')
    print(f'{"─"*50}')
    for cat, key_list in categories.items():
        print(f'  {cat:15s} | {key_list}')


In [41]:
show_types('NH')

📄 NH
──────────────────────────────────────────────────
  ham_BK          | ['ham_BK-10', 'ham_BK-12', 'ham_BK-14', 'ham_BK-36', 'ham_BK-8']
  ham_JW          | ['ham_JW-10', 'ham_JW-12', 'ham_JW-14', 'ham_JW-36', 'ham_JW-8']
  ham_molec       | ['ham_molec-10', 'ham_molec-12', 'ham_molec-14', 'ham_molec-36', 'ham_molec-8']
  ham_parity      | ['ham_parity-10', 'ham_parity-12', 'ham_parity-14', 'ham_parity-36', 'ham_parity-8']
